# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanzina-Aranya-Islam/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

The main signals are highly concentrated near zero, while a smaller number of observations have much larger values. For example, gsc_impressions has a median of 0 but a maximum of 40,084, and gsc_clicks has a median of 0 but a maximum of 274. ga4_sessions and sessions_ai also have many zero values. This indicates strongly skewed, heavy-tailed distributions, so averages alone may not represent a typical page well.

In [11]:
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')

path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df_march = pd.read_parquet(
    path,
    storage_options={"token": HF_TOKEN}
)

print("Rows:", len(df_march))
print("Columns:", df_march.columns.tolist())

Rows: 9841378
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Look at the basic distribution of the main numeric signals

features = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_sessions",
    "gsc_avg_position",
    "sessions_ai"
]

print(df_march[features].describe().T)

                      count       mean         std  min      25%  50%   75%  \
gsc_impressions   9841378.0  28.518119  155.926569  0.0  0.00000  0.0   6.0   
gsc_clicks        9841378.0   0.083508    0.781434  0.0  0.00000  0.0   0.0   
ga4_sessions      6822637.0   0.190514    1.968750  0.0  0.00000  0.0   0.0   
gsc_avg_position  3611061.0  15.826651   19.856034  0.0  3.74212  7.5  20.2   
sessions_ai       6822637.0   0.001306    0.056487  0.0  0.00000  0.0   0.0   

                      max  
gsc_impressions   40084.0  
gsc_clicks          274.0  
ga4_sessions        792.0  
gsc_avg_position    498.0  
sessions_ai          28.0  


## 2. Signal test #1 / #2 / #3 (verdict each)

Signal Test #1 — Impressions and clicks

Test: I tested whether pages with more GSC impressions also tend to have more GSC clicks.

Observed result: The Pearson correlation between gsc_impressions and gsc_clicks was 0.597, indicating a moderate positive relationship in the March 2026 data.

Verdict: CONFIRMED

The data supports the directional signal that pages with more search impressions tend to receive more clicks. This is an observed relationship, not evidence that increasing impressions alone will cause more clicks.

Signal Test #2 — Search position and clicks

Test: I tested whether better search position is associated with more clicks.

Observed result: The correlation between gsc_avg_position and gsc_clicks was -0.073, which is a very weak negative relationship.

Verdict: MIXED

The direction is consistent with the idea that better search position may be associated with more clicks, but the relationship is too weak in this dataset to treat the signal as strong evidence. This should be treated as a directional observation rather than a reliable rule.

Signal Test #3 — GA4 sessions and clicks

Test: I tested whether pages with more GSC clicks also tend to have more GA4 sessions.

Observed result: The Pearson correlation between ga4_sessions and gsc_clicks was 0.309, indicating a positive relationship in the March 2026 data.

Verdict: CONFIRMED

The data supports a directional relationship between GSC clicks and GA4 sessions. Pages with more clicks tend to have more sessions, although the relationship is not strong enough to treat clicks as a direct predictor of sessions. This is an observed relationship, not evidence that more clicks alone will cause more sessions.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal Test #1:
# Do pages with more impressions also tend to have more clicks?
test_df = df_march[
    ["gsc_impressions", "gsc_clicks"]
].dropna()

correlation = test_df["gsc_impressions"].corr(
    test_df["gsc_clicks"]
)

print("Impressions-clicks correlation:", correlation)

# Signal Test #2:
# Does better search position tend to be associated with more clicks?
test_df = df_march[
    ["gsc_avg_position", "gsc_clicks"]
].dropna()

correlation = test_df["gsc_avg_position"].corr(
    test_df["gsc_clicks"]
)

print("Position-clicks correlation:", correlation)

# Signal Test #3:
# Do pages with more GA4 sessions also tend to have more GSC clicks?

test_df = df_march[
    ["ga4_sessions", "gsc_clicks"]
].dropna()

correlation = test_df["ga4_sessions"].corr(
    test_df["gsc_clicks"]
)

print("GA4 sessions-clicks correlation:", correlation)

Impressions-clicks correlation: 0.5966873085053721
Position-clicks correlation: -0.07262796804626172
GA4 sessions-clicks correlation: 0.30895924553480325


## 3. The flag-linked test

Signal tested: GSC data availability.

Test: I compared GSC impressions and clicks between rows where GSC data was available and rows where it was unavailable.

Observed result: GSC-available rows had an average of 77.72 impressions and 0.228 clicks. Rows without GSC data had 0 impressions and 0 clicks.

Verdict: CONFIRMED

The data supports the expected relationship between GSC data availability and GSC performance fields: when GSC data is unavailable, impressions and clicks are also unavailable/zero in this dataset. This means GSC-based signals should only be interpreted for rows where GSC data is available. The result does not show that GSC availability itself causes better performance.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


available = df_march[df_march["gsc_data_available"] == True]
unavailable = df_march[df_march["gsc_data_available"] == False]

print("GSC available rows:", len(available))
print("GSC unavailable rows:", len(unavailable))

print("\nAverage impressions:")
print("Available:", available["gsc_impressions"].mean())
print("Unavailable:", unavailable["gsc_impressions"].mean())

print("\nAverage clicks:")
print("Available:", available["gsc_clicks"].mean())
print("Unavailable:", unavailable["gsc_clicks"].mean())

GSC available rows: 3611061
GSC unavailable rows: 6230317

Average impressions:
Available: 77.72164164493482
Unavailable: 0.0

Average clicks:
Available: 0.22758740436674982
Unavailable: 0.0


## 4. What this means in practice

Content teams should not treat missing GSC data as zero search performance. GSC-based signals are meaningful only when GSC data is available, so recommendations should account for data availability before comparing pages. The observed relationships are useful for prioritization, but they should not be treated as proof that changing one metric will cause another to improve.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.